# GraphCast Inference Pipeline

Two operating modes:

| Mode | Purpose | ERA5 download |
|------|---------|---------------|
| **`verify`** | Compare predictions against ERA5 ground truth | Input (2 steps) + all rollout steps |
| **`forecast`** | Pure forecasting (no future data needed) | Input (2 steps) only |

## Setup

1. **Checkpoint** — download from `gs://dm_graphcast/` (see cell below for exact path)
2. **Normalization stats** — 3 `.nc` files from the same bucket under `stats/`
3. **ERA5** — downloaded automatically via CDS API

### How the batch construction works

GraphCast needs a time axis with `2 + rollout_steps` entries:
- Steps 0–1: real ERA5 data (t-12h, t-6h relative to `init_time`)
- Steps 2+: either real ERA5 (verify mode) or NaN placeholders (forecast mode)

`extract_input_target_times` then shifts the time axis so the last input (step 1) maps to lead time 0, and all subsequent steps become positive lead times (6h, 12h, ...). Forcings (solar, year/day progress) are computed analytically by the library — no need for future ERA5 solar data.

In [ ]:
# stdlib
import dataclasses
import datetime
import functools
import math
from pathlib import Path
from typing import Optional

# third-party
import cdsapi
import haiku as hk
import jax
import numpy as np
import xarray

# graphcast
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout

In [ ]:
class GraphCastPipeline:
    """GraphCast inference pipeline with two modes: 'verify' and 'forecast'.

    verify  — downloads ERA5 for all rollout steps; allows comparison with ground truth.
    forecast — downloads only 2 input timesteps; pure autoregressive prediction.
    """

    PRESSURE_LEVELS = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]

    # CDS long names for the ERA5 API request
    PRESSURE_VARS_CDS = [
        "geopotential", "specific_humidity", "temperature",
        "u_component_of_wind", "v_component_of_wind", "vertical_velocity",
    ]
    SURFACE_VARS_CDS = [
        "2m_temperature", "mean_sea_level_pressure",
        "10m_u_component_of_wind", "10m_v_component_of_wind",
    ]
    STATIC_VARS_CDS = ["geopotential", "land_sea_mask"]

    # CDS short name -> GraphCast internal name
    CDS_TO_GC = {
        "z": "geopotential", "q": "specific_humidity", "t": "temperature",
        "u": "u_component_of_wind", "v": "v_component_of_wind", "w": "vertical_velocity",
        "t2m": "2m_temperature", "msl": "mean_sea_level_pressure",
        "u10": "10m_u_component_of_wind", "v10": "10m_v_component_of_wind",
        "tisr": "toa_incident_solar_radiation", "lsm": "land_sea_mask",
    }

    def __init__(
        self,
        ckpt_path: str,
        stats_dir: str,
        era5_dir: str = "era5_data",
        output_dir: str = "outputs",
    ):
        self.ckpt_path = Path(ckpt_path)
        self.stats_dir = Path(stats_dir)
        self.era5_dir = Path(era5_dir)
        self.output_dir = Path(output_dir)

        # Loaded lazily
        self._params = None
        self._state = None
        self._model_config = None
        self._task_config = None
        self._diffs_stddev = None
        self._mean = None
        self._stddev = None
        self._run_forward_jitted = None

    # -------------------------------------------------------------------------
    # Model loading (done once, reused across forecasts)
    # -------------------------------------------------------------------------

    def load_model(self):
        """Load checkpoint, stats, and JIT-compile the forward function."""
        print("[1/3] Loading checkpoint ...")
        with open(self.ckpt_path, "rb") as f:
            ckpt = checkpoint.load(f, graphcast.CheckPoint)
        self._params = ckpt.params
        self._state = {}
        self._model_config = ckpt.model_config
        self._task_config = ckpt.task_config
        print(f"  Model: {ckpt.description}")
        print(f"  Config: {self._model_config}")
        print(f"  Task:   {self._task_config}")

        print("[2/3] Loading normalization stats ...")
        self._diffs_stddev = xarray.load_dataset(self.stats_dir / "diffs_stddev_by_level.nc").compute()
        self._mean = xarray.load_dataset(self.stats_dir / "mean_by_level.nc").compute()
        self._stddev = xarray.load_dataset(self.stats_dir / "stddev_by_level.nc").compute()

        print("[3/3] Building JIT-compiled forward function ...")
        self._run_forward_jitted = self._build_jitted_forward()
        print("Model ready.")

    def _build_jitted_forward(self):
        diffs_stddev = self._diffs_stddev
        mean = self._mean
        stddev = self._stddev
        model_config = self._model_config
        task_config = self._task_config
        params = self._params
        state = self._state

        def construct_wrapped_graphcast(mc, tc):
            predictor = graphcast.GraphCast(mc, tc)
            predictor = casting.Bfloat16Cast(predictor)
            predictor = normalization.InputsAndResiduals(
                predictor,
                diffs_stddev_by_level=diffs_stddev,
                mean_by_level=mean,
                stddev_by_level=stddev,
            )
            predictor = autoregressive.Predictor(predictor, gradient_checkpointing=False)
            return predictor

        @hk.transform_with_state
        def run_forward(mc, tc, inputs, targets_template, forcings):
            predictor = construct_wrapped_graphcast(mc, tc)
            return predictor(inputs, targets_template=targets_template, forcings=forcings)

        forward_apply = functools.partial(run_forward.apply, model_config=model_config, task_config=task_config)
        forward_apply = functools.partial(forward_apply, params=params, state=state)
        forward_apply_jitted = jax.jit(forward_apply)

        def _call(rng, inputs, targets_template, forcings):
            preds, _ = forward_apply_jitted(rng=rng, inputs=inputs, targets_template=targets_template, forcings=forcings)
            return preds

        return _call

    # -------------------------------------------------------------------------
    # ERA5 download
    # -------------------------------------------------------------------------

    def _download_era5(self, init_time: datetime.datetime, rollout_steps: int, mode: str) -> dict:
        """Download ERA5 data via CDS API.

        In 'forecast' mode: only the 2 input timesteps (t-12h, t-6h).
        In 'verify' mode:   input timesteps + all rollout timesteps for ground truth comparison.
        """
        self.era5_dir.mkdir(parents=True, exist_ok=True)
        c = cdsapi.Client()

        t_m12 = init_time - datetime.timedelta(hours=12)
        t_m6 = init_time - datetime.timedelta(hours=6)

        # Input times: always needed
        input_dates = sorted({t_m12.strftime("%Y-%m-%d"), t_m6.strftime("%Y-%m-%d")})
        input_times = sorted({t_m12.strftime("%H:%M"), t_m6.strftime("%H:%M")})

        common_input = {
            "product_type": "reanalysis",
            "area": [90, -180, -90, 180],
            "grid": [0.25, 0.25],
            "format": "netcdf",
            "date": f"{input_dates[0]}/{input_dates[-1]}",
            "time": input_times,
        }

        def _dl(label, path, dataset, request):
            if path.exists():
                print(f"  [SKIP] {label} -> {path}")
                return
            print(f"  [DOWNLOAD] {label} -> {path}")
            c.retrieve(dataset, request, str(path))

        # Pressure-level variables (input times)
        pl_path = self.era5_dir / "era5_pressure_levels.nc"
        _dl("Pressure levels (input)", pl_path, "reanalysis-era5-pressure-levels", {
            **common_input,
            "variable": self.PRESSURE_VARS_CDS,
            "pressure_level": [str(p) for p in self.PRESSURE_LEVELS],
        })

        # Surface variables (input times)
        sl_path = self.era5_dir / "era5_surface.nc"
        _dl("Surface (input)", sl_path, "reanalysis-era5-single-levels", {
            **common_input,
            "variable": self.SURFACE_VARS_CDS,
        })

        # Static variables (single snapshot)
        static_path = self.era5_dir / "era5_static.nc"
        _dl("Static", static_path, "reanalysis-era5-single-levels", {
            "product_type": "reanalysis",
            "area": [90, -180, -90, 180],
            "grid": [0.25, 0.25],
            "format": "netcdf",
            "variable": self.STATIC_VARS_CDS,
            "date": t_m6.strftime("%Y-%m-%d"),
            "time": [t_m6.strftime("%H:%M")],
        })

        nc_paths = {
            "pressure_levels": pl_path,
            "surface": sl_path,
            "static": static_path,
        }

        # In verify mode, also download ERA5 for the rollout period (ground truth)
        if mode == "verify":
            rollout_times_dt = [
                init_time + datetime.timedelta(hours=6 * i)
                for i in range(rollout_steps)
            ]
            rollout_dates = sorted({t.strftime("%Y-%m-%d") for t in rollout_times_dt})
            rollout_hours = sorted({t.strftime("%H:%M") for t in rollout_times_dt})

            common_rollout = {
                "product_type": "reanalysis",
                "area": [90, -180, -90, 180],
                "grid": [0.25, 0.25],
                "format": "netcdf",
                "date": f"{rollout_dates[0]}/{rollout_dates[-1]}",
                "time": rollout_hours,
            }

            gt_pl_path = self.era5_dir / "era5_pressure_levels_groundtruth.nc"
            _dl("Pressure levels (ground truth)", gt_pl_path, "reanalysis-era5-pressure-levels", {
                **common_rollout,
                "variable": self.PRESSURE_VARS_CDS,
                "pressure_level": [str(p) for p in self.PRESSURE_LEVELS],
            })

            gt_sl_path = self.era5_dir / "era5_surface_groundtruth.nc"
            _dl("Surface (ground truth)", gt_sl_path, "reanalysis-era5-single-levels", {
                **common_rollout,
                "variable": self.SURFACE_VARS_CDS,
            })

            nc_paths["pressure_levels_gt"] = gt_pl_path
            nc_paths["surface_gt"] = gt_sl_path

        print("ERA5 download complete.")
        return nc_paths

    # -------------------------------------------------------------------------
    # Batch construction
    # -------------------------------------------------------------------------

    def _prep_ds(self, ds: xarray.Dataset, t_zero: np.datetime64) -> xarray.Dataset:
        """Rename CDS conventions to GraphCast conventions, fix coords."""
        ds = ds.rename({k: v for k, v in self.CDS_TO_GC.items() if k in ds.data_vars})
        ds = ds.rename({k: v for k, v in {
            "latitude": "lat", "longitude": "lon",
            "pressure_level": "level", "valid_time": "time",
        }.items() if k in ds.dims or k in ds.coords})

        if "time" in ds.coords:
            ds = ds.assign_coords(time=ds["time"].values.astype("datetime64[ns]") - t_zero)
        if "lat" in ds.dims and ds["lat"].values[0] > ds["lat"].values[-1]:
            ds = ds.isel(lat=slice(None, None, -1))
        if "lon" in ds.dims and float(ds["lon"].min()) < 0:
            ds = ds.assign_coords(lon=ds["lon"] % 360).sortby("lon")
        ds = ds.assign_coords(
            lat=ds["lat"].astype(np.float32),
            lon=ds["lon"].astype(np.float32),
        )
        if "level" in ds.coords:
            ds = ds.assign_coords(level=ds["level"].astype(np.int32))
        return ds

    def _build_batch(
        self,
        nc_paths: dict,
        init_time: datetime.datetime,
        rollout_steps: int,
        mode: str,
    ) -> xarray.Dataset:
        """Build the xarray batch for GraphCast.

        In both modes the batch has shape (batch=1, time=2+rollout_steps, ...).
        - 'forecast': steps 2+ are NaN placeholders (become targets_template)
        - 'verify':   steps 2+ contain real ERA5 (used for ground truth comparison after rollout)
        """
        # t_zero: the reference epoch for the timedelta time axis.
        # We set it to init_time - 12h so that our two input steps sit at timedelta [0h, 6h].
        t_zero = np.datetime64(init_time - datetime.timedelta(hours=12), "ns")
        JUNK = ["number", "expver"]

        def _load(key):
            return xarray.load_dataset(nc_paths[key], drop_variables=JUNK).compute()

        # Load and prep input data (2 timesteps: t-12h, t-6h)
        ds_pl = self._prep_ds(_load("pressure_levels"), t_zero)
        ds_sl = self._prep_ds(_load("surface"), t_zero)

        # Sanity check: no NaN in input data
        for label, ds in [("pressure_levels", ds_pl), ("surface", ds_sl)]:
            for var in ds.data_vars:
                n = int(np.isnan(ds[var].values).sum())
                if n > 0:
                    raise ValueError(f"NaN in '{var}' ({label}): {n} values — check download")

        ds_input = xarray.merge([ds_pl, ds_sl], join="inner")
        assert ds_input.sizes["time"] == 2, f"Expected 2 input timesteps, got {ds_input.sizes['time']}"

        # Build future time slots
        future_times = np.array([
            np.timedelta64((i + 2) * 6, "h") for i in range(rollout_steps)
        ], dtype="timedelta64[ns]")

        if mode == "verify" and "pressure_levels_gt" in nc_paths:
            # Use real ERA5 for rollout timesteps
            ds_gt_pl = self._prep_ds(_load("pressure_levels_gt"), t_zero)
            ds_gt_sl = self._prep_ds(_load("surface_gt"), t_zero)
            ds_gt = xarray.merge([ds_gt_pl, ds_gt_sl], join="inner")

            # Select only the times matching our expected future_times
            ds_gt = ds_gt.sel(time=future_times)
            assert ds_gt.sizes["time"] == rollout_steps, (
                f"Expected {rollout_steps} ground truth steps, got {ds_gt.sizes['time']}"
            )
            ds = xarray.concat([ds_input, ds_gt], dim="time")
        else:
            # Forecast mode: NaN placeholders for future steps.
            # These become targets_template (wiped to NaN) and forcings (computed analytically).
            future_data = {
                var: xarray.DataArray(
                    np.full(
                        tuple(len(future_times) if d == "time" else ds_input.sizes[d] for d in ds_input[var].dims),
                        np.nan, dtype=np.float32,
                    ),
                    dims=ds_input[var].dims,
                    coords={d: (future_times if d == "time" else ds_input.coords[d])
                            for d in ds_input[var].dims if d in ds_input.coords or d == "time"},
                )
                for var in ds_input.data_vars
            }
            ds = xarray.concat([ds_input, xarray.Dataset(future_data)], dim="time")

        assert ds.sizes["time"] == 2 + rollout_steps

        # Static fields (no time dim)
        ds_st = self._prep_ds(_load("static"), t_zero).isel(time=0, drop=True)
        if "geopotential" in ds_st.data_vars:
            ds_st = ds_st.rename({"geopotential": "geopotential_at_surface"})

        # Add batch dim + static vars
        ds = ds.expand_dims("batch", axis=0)
        for var in ds_st.data_vars:
            ds[var] = ds_st[var]

        # Precipitation placeholder (target-only, not an input for operational model)
        if "total_precipitation_6hr" not in ds.data_vars:
            ds["total_precipitation_6hr"] = xarray.zeros_like(ds["2m_temperature"])

        # datetime coord (batch, time) — required by data_utils.add_derived_vars
        abs_times = (t_zero + ds["time"].values).astype("datetime64[ns]")
        ds = ds.assign_coords(datetime=(("batch", "time"), abs_times[np.newaxis, :]))

        print(f"  Batch shape: {dict(ds.sizes)}")
        print(f"  Time axis:   {ds['time'].values}")
        print(f"  Variables:   {sorted(ds.data_vars)}")
        return ds

    # -------------------------------------------------------------------------
    # Prepare inputs / targets / forcings
    # -------------------------------------------------------------------------

    def _prepare_inputs(self, batch: xarray.Dataset, rollout_steps: int):
        """Split batch into (inputs, targets_template, forcings) via data_utils."""
        inputs, targets, forcings = data_utils.extract_inputs_targets_forcings(
            batch,
            target_lead_times=slice("6h", f"{rollout_steps * 6}h"),
            **dataclasses.asdict(self._task_config),
        )
        targets_template = targets * np.nan

        print(f"  inputs:           {dict(inputs.sizes)}")
        print(f"  targets_template: {dict(targets_template.sizes)}")
        print(f"  forcings:         {dict(forcings.sizes)}")

        # Validate no NaN in inputs — NaN here propagates to all predictions
        for var in inputs.data_vars:
            n = int(np.isnan(inputs[var].values).sum())
            if n > 0:
                raise ValueError(f"NaN in input '{var}': {n} values")
        print("  Inputs are clean (no NaN).")
        return inputs, targets_template, forcings

    # -------------------------------------------------------------------------
    # Rollout
    # -------------------------------------------------------------------------

    def _run_rollout(self, inputs, targets_template, forcings) -> xarray.Dataset:
        """Run autoregressive rollout via chunked_prediction."""
        print("  Running rollout (first chunk includes JIT compile time) ...")
        predictions = rollout.chunked_prediction(
            self._run_forward_jitted,
            rng=jax.random.PRNGKey(0),
            inputs=inputs,
            targets_template=targets_template,
            forcings=forcings,
        )
        print(f"  Rollout complete: {dict(predictions.sizes)}")
        return predictions

    # -------------------------------------------------------------------------
    # Save / Load results
    # -------------------------------------------------------------------------

    def save_results(
        self,
        predictions: xarray.Dataset,
        init_time: datetime.datetime,
        ground_truth: xarray.Dataset = None,
        metrics: dict = None,
    ) -> Path:
        """Save predictions (and optionally ground truth + metrics) to NetCDF.

        Directory structure:
            {output_dir}/{YYYYMMDD_HH}/
                predictions.nc
                ground_truth.nc   (if provided)
                metrics.nc        (if provided)

        Returns the run directory path.
        """
        run_dir = self.output_dir / init_time.strftime("%Y%m%d_%H")
        run_dir.mkdir(parents=True, exist_ok=True)

        pred_path = run_dir / "predictions.nc"
        predictions.to_netcdf(pred_path)
        print(f"  Saved predictions  -> {pred_path}")

        if ground_truth is not None:
            gt_path = run_dir / "ground_truth.nc"
            ground_truth.to_netcdf(gt_path)
            print(f"  Saved ground truth -> {gt_path}")

        if metrics is not None:
            # Merge per-variable metric datasets, prefixing var name to avoid collisions
            merged = xarray.Dataset()
            for var, ds in metrics.items():
                for metric_name in ds.data_vars:
                    merged[f"{var}__{metric_name}"] = ds[metric_name]
            metrics_path = run_dir / "metrics.nc"
            merged.to_netcdf(metrics_path)
            print(f"  Saved metrics      -> {metrics_path}")

        return run_dir

    @staticmethod
    def load_results(run_dir: str) -> dict:
        """Load saved results from a run directory.

        Returns
        -------
        dict with keys 'predictions', and optionally 'ground_truth', 'metrics'.
        """
        run_dir = Path(run_dir)
        results = {}

        pred_path = run_dir / "predictions.nc"
        if pred_path.exists():
            results["predictions"] = xarray.load_dataset(pred_path)
            print(f"  Loaded predictions  <- {pred_path}")

        gt_path = run_dir / "ground_truth.nc"
        if gt_path.exists():
            results["ground_truth"] = xarray.load_dataset(gt_path)
            print(f"  Loaded ground truth <- {gt_path}")

        metrics_path = run_dir / "metrics.nc"
        if metrics_path.exists():
            raw = xarray.load_dataset(metrics_path)
            # Reconstruct per-variable dict: "varname__rmse" -> metrics["varname"]["rmse"]
            metrics = {}
            for full_name in raw.data_vars:
                var, metric_name = full_name.split("__", 1)
                if var not in metrics:
                    metrics[var] = xarray.Dataset()
                metrics[var][metric_name] = raw[full_name]
            results["metrics"] = metrics
            print(f"  Loaded metrics      <- {metrics_path}")

        return results

    # -------------------------------------------------------------------------
    # Public API
    # -------------------------------------------------------------------------

    def predict(
        self,
        init_time: datetime.datetime,
        rollout_steps: int = 40,
        mode: str = "forecast",
        skip_download: bool = False,
    ) -> xarray.Dataset:
        """Run a GraphCast forecast.

        Parameters
        ----------
        init_time : datetime.datetime
            Forecast initialization time (must be 00/06/12/18 UTC).
        rollout_steps : int
            Number of 6-hour steps to predict (40 = 10 days).
        mode : str
            'forecast' — only downloads 2 input steps, pure prediction.
            'verify'   — downloads all steps, enables ground truth comparison.
        skip_download : bool
            If True, reuse existing ERA5 files in era5_dir.

        Returns
        -------
        predictions : xarray.Dataset
        """
        assert mode in ("forecast", "verify"), f"mode must be 'forecast' or 'verify', got '{mode}'"
        assert self._run_forward_jitted is not None, "Call load_model() first"

        # Step 1: Download ERA5
        if not skip_download:
            print(f"\n[STEP 1] Downloading ERA5 (mode={mode}) ...")
            nc_paths = self._download_era5(init_time, rollout_steps, mode)
        else:
            print(f"\n[STEP 1] Skipping download, using existing files in {self.era5_dir}")
            nc_paths = {
                "pressure_levels": self.era5_dir / "era5_pressure_levels.nc",
                "surface": self.era5_dir / "era5_surface.nc",
                "static": self.era5_dir / "era5_static.nc",
            }
            if mode == "verify":
                nc_paths["pressure_levels_gt"] = self.era5_dir / "era5_pressure_levels_groundtruth.nc"
                nc_paths["surface_gt"] = self.era5_dir / "era5_surface_groundtruth.nc"

        # Step 2: Build batch
        print(f"\n[STEP 2] Building xarray batch ...")
        batch = self._build_batch(nc_paths, init_time, rollout_steps, mode)

        # Step 3: Prepare inputs
        print(f"\n[STEP 3] Preparing inputs / targets / forcings ...")
        inputs, targets_template, forcings = self._prepare_inputs(batch, rollout_steps)

        # Step 4: Rollout
        print(f"\n[STEP 4] Autoregressive rollout ({rollout_steps} steps = {rollout_steps * 6}h) ...")
        predictions = self._run_rollout(inputs, targets_template, forcings)

        return predictions

    def load_ground_truth(
        self,
        init_time: datetime.datetime,
        rollout_steps: int,
    ) -> xarray.Dataset:
        """Load ERA5 ground truth for the rollout period (for verification).

        Returns an xarray.Dataset with the same variable names and grid as predictions,
        with a 'time' coord of lead-time timedeltas [6h, 12h, ...].
        """
        t_zero = np.datetime64(init_time - datetime.timedelta(hours=12), "ns")
        JUNK = ["number", "expver"]

        gt_pl = xarray.load_dataset(self.era5_dir / "era5_pressure_levels_groundtruth.nc", drop_variables=JUNK).compute()
        gt_sl = xarray.load_dataset(self.era5_dir / "era5_surface_groundtruth.nc", drop_variables=JUNK).compute()

        gt_pl = self._prep_ds(gt_pl, t_zero)
        gt_sl = self._prep_ds(gt_sl, t_zero)
        gt = xarray.merge([gt_pl, gt_sl], join="inner")

        # Convert time axis to lead times matching predictions (6h, 12h, ...)
        # The ground truth times are at offsets [12h, 18h, 24h, ...] from t_zero.
        # Predictions have lead times [6h, 12h, ...] from the last input (t_zero + 6h).
        # So lead_time = gt_time - 6h  (since last input is at timedelta 6h from t_zero)
        gt = gt.assign_coords(time=gt["time"].values - np.timedelta64(6, "h"))

        return gt

In [ ]:
def compute_verification_metrics(predictions: xarray.Dataset, ground_truth: xarray.Dataset, variables=None):
    """Compute RMSE and ACC (anomaly correlation coefficient) per variable per lead time.

    Parameters
    ----------
    predictions : xarray.Dataset
        Output from pipeline.predict() — has dims (batch, time, lat, lon) or (batch, time, level, lat, lon).
    ground_truth : xarray.Dataset
        Output from pipeline.load_ground_truth() — same grid, same time coord.
    variables : list of str, optional
        Which variables to evaluate. Defaults to all shared variables.

    Returns
    -------
    metrics : dict[str, xarray.Dataset]
        Keys are variable names, values have coords (time,) or (time, level) with data vars 'rmse' and 'acc'.
    """
    if variables is None:
        variables = sorted(set(predictions.data_vars) & set(ground_truth.data_vars))

    # Latitude-based weights for area-weighted metrics
    lat = predictions.coords["lat"]
    weights = np.cos(np.deg2rad(lat))
    weights = weights / weights.mean()

    metrics = {}
    for var in variables:
        pred = predictions[var]
        truth = ground_truth[var]
        if "batch" in pred.dims:
            pred = pred.isel(batch=0)

        # Align on shared times
        shared_times = np.intersect1d(pred.coords["time"].values, truth.coords["time"].values)
        pred = pred.sel(time=shared_times)
        truth = truth.sel(time=shared_times)

        spatial_dims = [d for d in pred.dims if d in ("lat", "lon")]

        # Weighted RMSE
        diff_sq = (pred - truth) ** 2
        wmean = (diff_sq * weights).mean(dim=spatial_dims)
        rmse = np.sqrt(wmean)

        # ACC: anomaly correlation coefficient
        # anomaly = value - climatological mean (here approximated by spatial mean of truth)
        truth_mean = (truth * weights).mean(dim=spatial_dims)
        pred_anom = pred - truth_mean
        truth_anom = truth - truth_mean
        num = (pred_anom * truth_anom * weights).mean(dim=spatial_dims)
        den = np.sqrt(
            (pred_anom ** 2 * weights).mean(dim=spatial_dims) *
            (truth_anom ** 2 * weights).mean(dim=spatial_dims)
        )
        acc = num / den

        ds = xarray.Dataset({"rmse": rmse, "acc": acc})
        metrics[var] = ds
        
    return metrics


def print_metrics_summary(metrics: dict, lead_hours=None):
    """Print a readable summary table of verification metrics."""
    if lead_hours is None:
        lead_hours = [24, 72, 120, 240]

    for var, ds in metrics.items():
        print(f"\n{'='*60}")
        print(f"  {var}")
        print(f"{'='*60}")
        times = ds.coords["time"].values
        has_level = "level" in ds.dims

        for lh in lead_hours:
            td = np.timedelta64(lh, "h")
            if td not in times:
                continue
            sel = ds.sel(time=td)
            if has_level:
                # Average across levels for summary
                rmse_val = float(sel["rmse"].mean())
                acc_val = float(sel["acc"].mean())
            else:
                rmse_val = float(sel["rmse"])
                acc_val = float(sel["acc"])
            print(f"  +{lh:>3d}h  |  RMSE: {rmse_val:8.3f}  |  ACC: {acc_val:.4f}")

## Configuration

Update paths below to match your environment:
- **HPC**: `/raid/apaudel/graphcast/...`
- **Local Mac**: wherever you downloaded the checkpoint and stats

In [ ]:
# --- UPDATE THESE PATHS ---
CKPT_PATH = (
    "/raid/apaudel/graphcast/params/"
    "GraphCast_operational - ERA5-HRES 1979-2021 - resolution 0.25 - "
    "pressure levels 13 - mesh 2to6 - precipitation output only.npz"
)
STATS_DIR = "/raid/apaudel/graphcast/stats/"
ERA5_DIR = "./era5_data"
OUTPUT_DIR = "./outputs"

pipeline = GraphCastPipeline(
    ckpt_path=CKPT_PATH,
    stats_dir=STATS_DIR,
    era5_dir=ERA5_DIR,
    output_dir=OUTPUT_DIR,
)
pipeline.load_model()

## Example 1: Verification mode

Download ERA5 for both input and rollout steps. Run the forecast, then compare against ground truth.

In [ ]:
init_time = datetime.datetime(2024, 1, 1, 0, 0)
rollout_steps = 40  # 40 x 6h = 10 days

predictions = pipeline.predict(
    init_time=init_time,
    rollout_steps=rollout_steps,
    mode="verify",
)

In [ ]:
# Load ground truth and compute metrics
ground_truth = pipeline.load_ground_truth(init_time, rollout_steps)

metrics = compute_verification_metrics(
    predictions, ground_truth,
    variables=["2m_temperature", "geopotential", "temperature", "u_component_of_wind"],
)
print_metrics_summary(metrics)

# Save everything to disk
run_dir = pipeline.save_results(predictions, init_time, ground_truth=ground_truth, metrics=metrics)
print(f"\nAll results saved to: {run_dir}")

### Reload saved results (no model needed)

You can load predictions, ground truth, and metrics from disk without loading the model. Useful for visualization on a different machine (e.g. your Mac).

In [ ]:
# Load from disk — works without loading the model
saved = GraphCastPipeline.load_results("./outputs/20240101_00")

preds_loaded = saved["predictions"]
gt_loaded = saved["ground_truth"]
metrics_loaded = saved["metrics"]

# Verify it round-trips correctly
print_metrics_summary(metrics_loaded)

## Example 2: Forecast mode (no future ERA5 needed)

Only downloads the 2 input timesteps. This is the mode you'd use for operational forecasting initialized every 24 hours.

In [ ]:
predictions_fc = pipeline.predict(
    init_time=datetime.datetime(2024, 1, 1, 0, 0),
    rollout_steps=40,
    mode="forecast",
)

# Save predictions (no ground truth in forecast mode)
pipeline.save_results(predictions_fc, datetime.datetime(2024, 1, 1, 0, 0))

## Example 3: Sequential forecasts (every 24h)

The model is loaded once; each `predict()` call reuses the JIT-compiled forward function.

In [ ]:
# Run forecasts initialized every 24h over a 3-day window
base_time = datetime.datetime(2024, 1, 1, 0, 0)
results = {}

for day_offset in range(3):
    t = base_time + datetime.timedelta(days=day_offset)
    print(f"\n{'#'*60}")
    print(f"# Forecast init: {t}")
    print(f"{'#'*60}")

    # Use a separate era5_dir per init time to avoid file collisions
    pipeline.era5_dir = Path(f"./era5_data/{t.strftime('%Y%m%d_%H')}")
    
    preds = pipeline.predict(
        init_time=t,
        rollout_steps=20,  # 5 days
        mode="forecast",
    )
    results[t] = preds

print(f"\nCompleted {len(results)} forecasts.")